In [1]:
import os
import sys
import numpy as np
from pathlib import Path

# Set one level up as project root|
if os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))

from app.config import get_settings
from ingestion.loaders import load_pdf_pages, chunk_pages
from ingestion.build_index import embedd_chunks, build_index

In [2]:
get_settings().openai_api_key[:8] + "..."

'sk-proj-...'

In [3]:
DATA_PATH = Path("../data/NVIDIAAn.pdf")
DATA_PATH

PosixPath('../data/NVIDIAAn.pdf')

- #### Check loading

In [4]:
%%time
pages = list(load_pdf_pages(path=DATA_PATH))
first_page = pages[0]
print(f"first page nr:{first_page.page_number}.")
print(f"Initial text:\n{first_page.text[:200]}.")

first page nr:1.
Initial text:
NVIDIA Announces Financial Results for Second Quarter
Fiscal 2024
Record revenue of $13.51 billion, up 88% from Q1, up 101% from year ago
Record Data Center revenue of $10.32 billion, up 141% from Q1,.
CPU times: user 1.73 s, sys: 5.12 ms, total: 1.73 s
Wall time: 1.74 s


- #### Check loading

In [5]:
%%time
chunks = list(chunk_pages(pages))
assert 500 < chunks[0].length <= 800, f"Bad length: {chunks[0].length}"

CPU times: user 122 μs, sys: 4 μs, total: 126 μs
Wall time: 130 μs


In [11]:
print(f"{len(pages)} pages and {len(chunks)} chunks")
print(f"sampel ID {chunks[-1].id}")
print(f"avg chunk len = {int(sum(c.length for c in chunks) / len(chunks))} chars")
print(f"Example chunk: {chunks[-1].text[:50]}...")
# assert all(c.page in range(1, len(pages)+1) for c in chunks)

9 pages and 45 chunks
sampel ID p9_c7
avg chunk len = 709 chars
Example chunk:  other
countries. Other company and product names ...


In [ ]:
DATA_PATH.stem

- #### Check embedding vectorization

In [ ]:
%%time
embeddings = embedd_chunks(chunks[:2])

In [ ]:
print(f"There are {len(embeddings)} embeddings")
print(
    f"Size of embedding for {get_settings().openai_embedding_model} model is {len(embeddings[0])} dimensions"
)  # Should be 1536
print(f"Embedding values range from {min(embeddings[0]):.3f} to {max(embeddings[0]):.3f}")
print(f"Example embedding: {[round(emb, 3) for emb in embeddings[0][:5]]}...")

In [ ]:
len("Long pages. ") * 100 // (500 - 100) + 